# 2. Évaluer D01 sans confondre abstention et absence

**Question.** Que signifient précision élevée et rappel faible dans les 32 essais historiques ?

A utilise les informations propres et le LiDAR ; B ajoute les messages réellement reçus. C conserve le diagnostic individuel B et ajoute le graphe. Les 32 essais ont été choisis pour le développement et la démonstration ; ils ne constituent pas un test indépendant de généralisation.


In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from cav_recovery.laboratory.workbench import Laboratory
from cav_recovery.laboratory.storage import json_file

lab = Laboratory.from_environment()
display(HTML(lab.table_html([{'résultats': str(lab.root), 'M02': str(lab.dataset)}])))


## Comptes, tâches et formules

Une vraie confirmation prédite est TP ; une confirmation prédite sans référence est FP ; une confirmation de référence non détectée, y compris avec abstention, est FN. Précision = TP/(TP+FP), rappel = TP/(TP+FN), F1 = 2TP/(2TP+FP+FN). Une précision sans prédiction positive reste non définie.

Les valeurs suivantes sont agrégées par la même fonction d'évaluation que la campagne, à partir de ses 32 fichiers de résultats. Les nombreux rendez-vous sont corrélés : l'unité expérimentale reste l'épisode.


In [ ]:
mesures = lab.historical_metrics()
display(HTML(lab.table_html([m for m in mesures if m['task'] in ('A', 'B')])))
display(HTML(lab.table_html(lab.historical_abstentions())))


## Pourquoi le rappel reste limité

Les reculs physiques, même faibles, peuvent sortir du domaine nominal K04. Une absence d'identification ou de preuve géométrique recevable impose également l'abstention. Le contrat n'est pas abaissé pour augmenter le score. Les 62 faux positifs de B restent dans les mesures, même si l'audit les situe sous le seuil surfacique M02.

Le correctif de l'horloge C-RDG refuse une preuve propre antidatée par le graphe. Il ne fournit pas de gain de rappel sur les flux historiques correctement datés. Les fixtures F01 A/B/C restent des tests de contrat distincts des configurations de comparaison A/B/C.


In [ ]:
evenements = [dict(essai=r['name'], **e) for r in lab.historical_episodes() for e in r['event_results']]
retrouves = [e for e in evenements if e['mode']=='B' and not e['missed']]
manques = [e for e in evenements if e['mode']=='B' and e['missed']]
display(HTML(lab.table_html([{'retrouvés B': len(retrouves), 'manqués B': len(manques),
    'délai médian conditionnel (s)': float(np.median([e['delay_s'] for e in retrouves])),
    'délai maximal conditionnel (s)': max(e['delay_s'] for e in retrouves)}])))


## Comparaison appariée avec les modèles temporels

La nouvelle sous-étude utilise exactement les mêmes endpoints, cibles disponibles maintenant et groupes tenus à part pour D01 et le TCN. La cadence est de 2 s sur des rendez-vous de preuve valides. Ces chiffres ne remplacent donc pas les anciens scores à 0,4 s sur les 32 essais.


In [ ]:
display(HTML(lab.table_html(lab.model_results('integrated'))))
fig = lab.plot_quality('integrated')
plt.show()
display(HTML(lab.table_html(lab.transfer_results())))
display(HTML(lab.table_html(lab.replay_costs())))


**Interprétation.** Une alerte statistique TCN peut couvrir des abstentions ou produire de nouvelles erreurs. Elle ne transforme pas une preuve K04 manquante en confirmation physique. La fusion « D01 B ou TCN » est une règle d'alerte statistique étudiée séparément, sans action de récupération.
